[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/00_intro/00_proyecto_intro.ipynb)

# Proyecto · Políglota v0: corpus, métricas y baseline BM25
**Módulo 00 · Intro, entorno e hilo conductor**

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración** | ≈ 1 h |
| **GPU** | Ninguna (CPU) |
| **Entregable** | `poliglota_v0.json` con el baseline medido (lo reutilizarás como punto de comparación en módulos posteriores) |

## Contexto de negocio
Eres ML engineer en **Políglota Labs**. La startup quiere lanzar un buscador para documentación técnica en español, inglés y código. Antes de gastar un euro en embeddings, tu CTO pide un **«Políglota v0»**: un buscador léxico (BM25) con un banco de evaluación pequeño pero fiable, para saber desde dónde partimos y qué tienen que superar los modelos futuros.

Tu misión: (1) cargar y entender el corpus, (2) implementar las métricas y el BM25 **tú mismo**, (3) medir el baseline por idioma con intervalo de confianza, (4) analizar dónde falla y (5) guardar el baseline.

**Conexión con módulos siguientes.** Este baseline se compara contra tu tokenizer propio (01–05, afecta a la tokenización léxica) y contra los embedders (09–13). La normalización Unicode (módulo 01) y el tokenizer BPE (módulo 02) cambiarán estos números.

## Dataset
Corpus Políglota (`carga_corpus`, ver lección 00): Wikipedia ES/EN (HF `wikimedia/wikipedia`) + código Python (CodeSearchNet), con una cadena **oficial → espejo → sintético**. En `FAST_DEV_RUN=True`: 150 artículos por idioma + 100 funciones y 110 consultas. La celda de carga te dirá si `source` es `real` o `synthetic`.

## Entregables y criterios de evaluación (rúbrica)

| # | Criterio | Umbral (corpus **sintético**, `FAST_DEV_RUN=True`) | Umbral (corpus **real**) |
|---|---|---|---|
| R1 | `estadisticas_corpus` devuelve las 3 filas y la suma de documentos correcta | tests ✅ | tests ✅ |
| R2 | `ndcg_at_k`, `recall_at_k`, `mrr_at_k` pasan los tests de casos límite (duplicados, sin relevantes, ejemplo a mano) | tests ✅ | tests ✅ |
| R3 | Tu `BM25` coincide con `rank_bm25.BM25Okapi` (idf=`okapi`, mismos $k_1$, $b$) | máx. \|Δscore\| ≤ 1e-6 | igual |
| R4 | nDCG@10 global de tu baseline | ≥ **0.82** | sin umbral calibrado*: reporta tu valor y compáralo con la solución |
| R5 | nDCG@10 por idioma (ES / EN) | ES ≥ **0.92**, EN ≥ **0.87** | sin umbral calibrado* |
| R6 | nDCG@10 en consultas de código con `split_ident` | ≥ **0.60** y mejora ≥ **+0.02** frente a `\w+` | no empeora frente a `\w+` |
| R7 | IC 95 % bootstrap calculado y `poliglota_v0.json` guardado con las claves requeridas | ✅ | ✅ |
| R8 | Análisis de errores escrito (≥ 3 hipótesis razonadas sobre las consultas que fallan) | manual | manual |

\* Los umbrales sintéticos son lo que **mide la solución de referencia** (nDCG@10 global = 0.878, ES = 0.982, EN = 0.932, código = 0.668; código con `\w+` = 0.606; n = 110 consultas, semilla 42) **menos un margen de ≈ 0.06**. No hemos podido medir la solución sobre Wikipedia real en el entorno de autoría (Hugging Face y Wikipedia estaban bloqueados), así que con datos reales no inventamos un umbral: la solución imprime sus valores al final del notebook para que compares.

## 0. Preparación (celdas dadas: no hay que modificarlas)

In [ ]:
!pip install -q numpy pandas matplotlib rank_bm25 regex

In [ ]:
import math, random, re, sys, time
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True      # True: subconjunto pequeño (minutos). False: corpus mayor.
CHECKS: dict = {}        # aquí se registra el resultado de cada comprobación de la rúbrica

def chequea(nombre, fn):
    """Ejecuta fn() y registra ✅/❌/⏳ sin romper «Ejecutar todo»."""
    try:
        fn(); CHECKS[nombre] = True; print("✅", nombre)
    except NotImplementedError as e:
        CHECKS[nombre] = None; print("⏳ pendiente:", nombre, "→", e)
    except AssertionError as e:
        CHECKS[nombre] = False; print("❌", nombre, "→", e)
    except Exception as e:  # noqa: BLE001
        CHECKS[nombre] = False; print("💥", nombre, "→", type(e).__name__, e)

In [ ]:
# ── carga_corpus (celda 1/3) ──
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── carga_corpus (celda 2/3) ──
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus (celda 3/3) ──
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# Utilidades dadas: tokenización léxica y ordenación determinista por score
import unicodedata
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)




def rank_from_scores(scores, doc_ids, k=None):
    """Orden por score desc.; los empates se rompen por orden del corpus (reproducible)."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    return [doc_ids[i] for i in (order[:k] if k else order)]

def buscar(bm, q_tokens, k=10):
    return rank_from_scores(bm.scores(q_tokens), bm.doc_ids, k)

def evaluate(run, qrels, k=10):
    """Promedia nDCG/Recall/MRR@k sobre las consultas con algún relevante (usa TUS funciones de métricas)."""
    n, tot = 0, dict(ndcg=0.0, recall=0.0, mrr=0.0)
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        tot["ndcg"] += ndcg_at_k(run.get(qid, []), rels, k); tot["recall"] += recall_at_k(run.get(qid, []), rels, k)
        tot["mrr"] += mrr_at_k(run.get(qid, []), rels, k); n += 1
    return {**{m: v / max(n, 1) for m, v in tot.items()}, "n_queries": n}

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, queries, qrels, source = corpus["docs"], corpus["queries"], corpus["qrels"], corpus["source"]
print(f"source = {source!r} | {len(docs)} docs | {len(queries)} queries")

## TODO 1 · Estadísticas del corpus
Implementa `estadisticas_corpus(docs)`: un `DataFrame` indexado por `lang` con `n_docs`, `chars_medio`, `bytes_medio`, `palabras_medio` y `bytes_por_char` (= `bytes_medio / chars_medio`). «Bytes» es la longitud del texto codificado en **UTF-8**.

<details>
<summary>🪜 Pista 1</summary>

Construye un `DataFrame` a partir de la lista de dicts y añade columnas con la longitud en caracteres, en bytes (`len(t.encode('utf-8'))`) y en palabras (`t.split()`). Después agrupa por `lang`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`df.groupby('lang').agg(n_docs=('id', 'count'), chars_medio=('chars', 'mean'), ...)`. Calcula `bytes_por_char` como cociente de dos columnas del resultado.

</details>

In [ ]:
def estadisticas_corpus(docs):
    # TODO: devuelve un DataFrame indexado por 'lang' con las 5 columnas indicadas
    raise NotImplementedError("TODO 1: implementa estadisticas_corpus")

def test_stats():
    g = estadisticas_corpus(docs)
    assert set(g.index) == {"es", "en", "code"}, "faltan idiomas en el índice"
    assert g.n_docs.sum() == len(docs), "n_docs no suma el total"
    assert (g.bytes_por_char >= 1).all(), "bytes_por_char no puede ser < 1 (un carácter ocupa ≥ 1 byte)"
chequea("R1 estadísticas", test_stats)
try:
    display(estadisticas_corpus(docs))
except NotImplementedError:
    pass

## TODO 2 · Métricas de retrieval
Implementa `ndcg_at_k`, `recall_at_k` y `mrr_at_k` (ganancia **lineal**, descuento $1/\log_2(i+1)$). Reglas: los **duplicados** en `ranked_ids` cuentan una vez; si no hay relevantes devuelven `0.0`; `rels` es `doc_id → relevancia` y relevante = `rel > 0`.

<details>
<summary>🪜 Pista 1</summary>

Primero deduplica `ranked_ids` conservando el orden (`list(dict.fromkeys(...))`) y recorta a `k`. El DCG suma `rel / log2(posición + 1)`; con índice desde 0 es `log2(i + 2)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

IDCG: ordena las relevancias `> 0` de mayor a menor, toma `k` y calcula el mismo DCG. nDCG = DCG / IDCG (o 0 si IDCG = 0). Recall@k = |relevantes ∩ top-k| / |relevantes|. MRR@k = 1/(posición del primer relevante).

</details>

In [ ]:
def ndcg_at_k(ranked_ids, rels, k=10):
    # TODO
    raise NotImplementedError("TODO 2a: implementa ndcg_at_k")

def recall_at_k(ranked_ids, rels, k=10):
    # TODO
    raise NotImplementedError("TODO 2b: implementa recall_at_k")

def mrr_at_k(ranked_ids, rels, k=10):
    # TODO
    raise NotImplementedError("TODO 2c: implementa mrr_at_k")

def test_metricas():
    r, rel = ["d3", "d1", "d7", "d2"], {"d1": 1, "d2": 1}
    assert abs(ndcg_at_k(r, rel, 4) - 0.6510) < 1e-3, "ejemplo a mano de la lección: nDCG@4 = 0.651"
    assert recall_at_k(r, rel, 4) == 1.0 and recall_at_k(r, rel, 1) == 0.0
    assert mrr_at_k(r, rel, 4) == 0.5 and mrr_at_k(r, rel, 1) == 0.0
    assert ndcg_at_k(["d1", "d1", "d1"], {"d1": 1}, 10) == 1.0, "los duplicados no deben inflar el DCG"
    assert ndcg_at_k(r, {}, 10) == 0.0 and recall_at_k(r, {}, 10) == 0.0 and mrr_at_k(r, {}, 10) == 0.0
    assert abs(ndcg_at_k(["b", "a"], {"a": 3, "b": 2}, 2) - (2 + 3 / math.log2(3)) / (3 + 2 / math.log2(3))) < 1e-9, "ganancia lineal con grados"
chequea("R2 métricas", test_metricas)

## TODO 3 · BM25 con índice invertido
Completa `fit` y `scores`. `fit` recibe `docs_tokens` (lista de listas de términos) y `doc_ids`; debe guardar `self.N`, `self.dl` (longitudes), `self.avgdl`, `self.postings` (`término → [(índice_doc, tf), ...]`) y `self.idf` (dict). Soporta `idf="lucene"` ($\ln(1+\frac{N-n+0.5}{n+0.5})$) e `idf="okapi"` ($\ln\frac{N-n+0.5}{n+0.5}$ con suelo $\epsilon\cdot\overline{\text{IDF}}$ para los negativos, $\epsilon=0.25$, como `rank_bm25`). `scores(q_tokens)` devuelve un `np.ndarray` de tamaño `N`.

<details>
<summary>🪜 Pista 1</summary>

Para cada documento cuenta los términos con `Counter(tokens)` y añade `(i, tf)` a `postings[término]`. La *document frequency* $n_t$ es `len(postings[t])`.

</details>

<details>
<summary>🪜 Pista 2</summary>

En `scores`, recorre solo los términos de la consulta y, para cada `(i, tf)` de su lista de postings, suma `idf[t] * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[i] / avgdl))` a `s[i]`. El suelo de `okapi`: `piso = epsilon * media(idf)`; los `idf < 0` pasan a valer `piso`.

</details>

In [ ]:
class BM25:
    def __init__(self, k1=1.2, b=0.75, idf="lucene", epsilon=0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens, doc_ids):
        # TODO: guarda self.doc_ids, self.N, self.dl, self.avgdl, self.postings y self.idf
        raise NotImplementedError("TODO 3a: implementa BM25.fit")

    def scores(self, q_tokens):
        # TODO: devuelve un array (N,) con el score BM25 de cada documento
        raise NotImplementedError("TODO 3b: implementa BM25.scores")

def test_bm25():
    toks = [["a", "b", "b"], ["b", "c"], ["c", "c", "d"], ["e"]]
    bm = BM25(k1=1.2, b=0.75).fit(toks, ["0", "1", "2", "3"])
    s = bm.scores(["b"])
    assert s.shape == (4,) and s[3] == 0 and s[2] == 0 and s[0] > s[1] > 0, "b aparece 2 veces en el doc 0 (más largo) y 1 en el doc 1"
    assert buscar(bm, ["c"], 2) == ["2", "1"], "c: tf=2 en el doc 2 gana al doc 1"
chequea("R3a BM25 mini", test_bm25)

**Validación contra `rank_bm25`.** La librería es la referencia de industria (versión sencilla). Con la misma fórmula de IDF y los mismos $k_1$, $b$ los scores deben coincidir.

In [ ]:
def test_vs_rank_bm25():
    from rank_bm25 import BM25Okapi
    toks = [tokenize_simple(d["text"]) for d in docs]
    ref = BM25Okapi(toks, k1=1.5, b=0.75)
    mio = BM25(k1=1.5, b=0.75, idf="okapi").fit(toks, [d["id"] for d in docs])
    dif = max(np.abs(ref.get_scores(q) - mio.scores(q)).max() for q in (tokenize_simple(x["text"]) for x in queries[:20]))
    assert dif <= 1e-6, f"máx |Δscore| = {dif:.2e} (> 1e-6): revisa la fórmula / el suelo de idf negativos"
chequea("R3 BM25 == rank_bm25", test_vs_rank_bm25)

## TODO 4 · Tokenización de código: dividir identificadores
En la lección vimos que dividir `snake_case` y `camelCase` ayuda en código. Implementa `split_ident(text)`: devuelve los términos de `tokenize_simple` aplicados al texto con los identificadores **divididos**, y además los identificadores enteros que contenían `_` (para no perder `os_path`). Ejemplo: `split_ident("getHTTPResponse_code")` ⊇ `['get', 'http', 'response', 'code']`.

<details>
<summary>🪜 Pista 1</summary>

Inserta un espacio entre minúscula/dígito y mayúscula (`re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', t)`), trata aparte las siglas (`HTTPResponse` → `HTTP Response`) y cambia `_` por espacio antes de `tokenize_simple`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Dos `re.sub` (camelCase y siglas) + `.replace('_', ' ')`; luego `tokenize_simple(...)`. Para conservar los identificadores con guion bajo, usa primero `[w for w in tokenize_simple(text) if '_' in w]`.

</details>

In [ ]:
def split_ident(text):
    # TODO
    raise NotImplementedError("TODO 4: implementa split_ident")

def test_split():
    assert {"get", "http", "response", "code"} <= set(split_ident("getHTTPResponse_code"))
    assert {"join", "path", "os_join_path"} <= set(split_ident("os_join_path(a, b)"))
    assert split_ident("Hola mundo") == ["hola", "mundo"]
chequea("R6a split_ident", test_split)

## TODO 5 · Construir el índice, buscar y medir por idioma
Construye **un único índice** con todos los documentos (así una consulta en español también compite contra documentos en inglés y código, como en el producto real). Tokeniza los documentos de tipo `code` con `split_ident` y el resto con `tokenize_simple`; las **consultas** se tokenizan siempre con `tokenize_simple` (son lenguaje natural, incluso las de código). Compara contra el baseline que usa `tokenize_simple` en todo y rellena el diccionario `RESULTADOS`.

<details>
<summary>🪜 Pista 1</summary>

Una función `tok_doc(d)` que mire `d['lang']`. `BM25().fit([tok_doc(d) for d in docs], [d['id'] for d in docs])`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`run = {q['qid']: buscar(bm, tokenize_simple(q['text']), 10) for q in queries}` y `evaluate(run, qrels_del_grupo)`, donde `qrels_del_grupo` filtra por `q['lang']`. Repite con el baseline sin `split_ident`.

</details>

In [ ]:
RESULTADOS = {}

def evaluar_grupo(run, lang=None):
    qr = {q["qid"]: qrels[q["qid"]] for q in queries if lang is None or q["lang"] == lang}
    return evaluate(run, qr)["ndcg"]

def pipeline(tok_doc):
    # TODO: construye el índice global con tok_doc, busca todas las consultas (top-10) y devuelve (bm, run)
    raise NotImplementedError("TODO 5: implementa pipeline")

try:
    bm_base, run_base = pipeline(lambda d: tokenize_simple(d["text"]))
    bm_split, run_split = pipeline(lambda d: split_ident(d["text"]) if d["lang"] == "code" else tokenize_simple(d["text"]))
    RESULTADOS = {"ndcg_all": evaluar_grupo(run_split), "ndcg_es": evaluar_grupo(run_split, "es"), "ndcg_en": evaluar_grupo(run_split, "en"),
                  "ndcg_code": evaluar_grupo(run_split, "code"), "ndcg_code_base": evaluar_grupo(run_base, "code")}
    print({k: round(v, 3) for k, v in RESULTADOS.items()})
except NotImplementedError as e:
    print("⏳ pendiente:", e)

In [ ]:
# Gráfico: baseline vs split_ident por grupo
if RESULTADOS:
    grupos = ["es", "en", "code"]
    a = [evaluar_grupo(run_base, g) for g in grupos]; b = [evaluar_grupo(run_split, g) for g in grupos]
    x = np.arange(3); fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.bar(x - 0.2, a, 0.4, label="tokenize_simple", color="#6B7280"); ax.bar(x + 0.2, b, 0.4, label="+ split_ident en código", color="#1F6FEB")
    ax.set_xticks(x, grupos); ax.set_ylim(0, 1.05); ax.set_ylabel("nDCG@10"); ax.set_title(f"Políglota v0 (corpus {source})"); ax.legend()
    plt.tight_layout(); plt.show()

## TODO 6 · Intervalo de confianza bootstrap
Implementa `bootstrap_ci(valores, n_boot=2000, alpha=0.05, seed=42)`: remuestrea **las consultas** con reemplazo `n_boot` veces y devuelve `(media, (lo, hi))` con los cuantiles `alpha/2` y `1-alpha/2` de las medias remuestreadas.

<details>
<summary>🪜 Pista 1</summary>

`rng = np.random.default_rng(seed)`; `rng.choice(valores, size=(n_boot, len(valores)), replace=True)` te da todos los remuestreos de golpe.

</details>

<details>
<summary>🪜 Pista 2</summary>

`medias = remuestreos.mean(axis=1)`; `np.quantile(medias, [alpha/2, 1-alpha/2])`.

</details>

In [ ]:
def bootstrap_ci(valores, n_boot=2000, alpha=0.05, seed=42):
    # TODO
    raise NotImplementedError("TODO 6: implementa bootstrap_ci")

def test_ci():
    v = np.array([0.0, 1.0] * 50)
    m, (lo, hi) = bootstrap_ci(v)
    assert abs(m - 0.5) < 1e-9 and 0.35 < lo < 0.5 < hi < 0.65, f"IC inesperado: {lo:.3f}-{hi:.3f}"
chequea("R7a bootstrap", test_ci)
if RESULTADOS:
    try:
        v = np.array([ndcg_at_k(run_split[q["qid"]], qrels[q["qid"]]) for q in queries])
        m, (lo, hi) = bootstrap_ci(v); RESULTADOS["ci95"] = [float(lo), float(hi)]
        print(f"nDCG@10 global = {m:.3f}  IC95% [{lo:.3f}, {hi:.3f}]  (n = {len(v)})")
    except NotImplementedError:
        pass

## TODO 7 · Análisis de errores
Lista las 5 consultas **de código** con peor resultado (la posición del relevante en el ranking completo, o «fuera del top-100»). Después, en la celda de texto siguiente, **escribe al menos 3 hipótesis** razonadas de por qué fallan (vocabulario, tokenización, idioma, ruido…), apoyándote en lo que ves.

<details>
<summary>🪜 Pista 1</summary>

Para cada consulta de código pide el ranking completo (`k=len(docs)`) y busca la posición del único relevante con `.index(...)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Ordena por esa posición descendente y muestra `q['text']`, la posición y los primeros 200 caracteres del documento relevante (`docs` por id).

</details>

In [ ]:
def peores_consultas(bm, tok_q=tokenize_simple, lang="code", n=5):
    # TODO: devuelve una lista de (qid, texto_consulta, posición_del_relevante) con las n peores, de peor a mejor
    raise NotImplementedError("TODO 7: implementa peores_consultas")

try:
    por_id = {d["id"]: d for d in docs}
    for qid, texto, pos in peores_consultas(bm_split):
        rel_id = next(iter(qrels[qid]))
        print(f"[{qid}] pos={pos} | consulta: {texto!r}\n      relevante: {por_id[rel_id]['text'][:160]!r}\n")
except NotImplementedError as e:
    print("⏳ pendiente:", e)
except NameError:
    print("⏳ completa antes el TODO 5")

**✍️ Tus hipótesis (escribe aquí, ≥ 3):**

1. …
2. …
3. …

## TODO 8 · Guardar el baseline
Guarda `poliglota_v0.json` con las claves `source`, `n_docs`, `n_queries`, `k1`, `b`, `tokenizacion`, `ndcg_all`, `ndcg_es`, `ndcg_en`, `ndcg_code`, `ndcg_code_base`, `ci95`. En módulos posteriores lo cargarás para comparar tu nuevo tokenizer/embedder con este punto de partida.

In [ ]:
import json, pathlib
CLAVES = ["source", "n_docs", "n_queries", "k1", "b", "tokenizacion", "ndcg_all", "ndcg_es", "ndcg_en", "ndcg_code", "ndcg_code_base", "ci95"]

def guardar_baseline(path="poliglota_v0.json"):
    # TODO: construye el dict con las CLAVES (valores de RESULTADOS + metadatos), guárdalo como JSON y devuélvelo
    raise NotImplementedError("TODO 8: implementa guardar_baseline")

def test_json():
    d = guardar_baseline()
    assert all(k in d for k in CLAVES), f"faltan claves: {[k for k in CLAVES if k not in d]}"
    assert json.loads(pathlib.Path("poliglota_v0.json").read_text())["source"] == source
chequea("R7b poliglota_v0.json", test_json)

## Rúbrica: comprobación automática
Ejecuta esta celda cuando hayas terminado. R8 (análisis de errores) es manual.

In [ ]:
UMBRALES_SINTETICO = {'all': 0.82, 'es': 0.92, 'en': 0.87, 'code': 0.6, 'mejora': 0.02}   # medidos con la solución (ver tabla de la rúbrica)

def rubrica():
    filas = [(k, {True: "✅", False: "❌", None: "⏳"}[v]) for k, v in CHECKS.items()]
    if RESULTADOS:
        u = UMBRALES_SINTETICO
        if source == "synthetic":
            reglas = [("R4 nDCG@10 global", RESULTADOS["ndcg_all"], u["all"]), ("R5 ES", RESULTADOS["ndcg_es"], u["es"]),
                      ("R5 EN", RESULTADOS["ndcg_en"], u["en"]), ("R6 código con split_ident", RESULTADOS["ndcg_code"], u["code"]),
                      ("R6 mejora vs \\w+", RESULTADOS["ndcg_code"] - RESULTADOS["ndcg_code_base"], u["mejora"])]
            filas += [(f"{n} = {v:.3f} (≥ {t:.2f})", "✅" if v >= t else "❌") for n, v, t in reglas]
        else:
            print("ℹ️ corpus REAL: no hay umbrales calibrados; reporta y compara con la solución.")
            filas.append(("R6 split_ident no empeora código", "✅" if RESULTADOS["ndcg_code"] >= RESULTADOS["ndcg_code_base"] else "❌"))
    for n, e in filas:
        print(f"{e}  {n}")
    print("\n(R8: análisis de errores escrito → revísalo tú)")
rubrica()

## Retos extra (nivel experto)
1. **Ajusta $k_1$ y $b$** con una búsqueda en rejilla sobre un *split de validación* de consultas y reporta en un split de test distinto (¿cuánto sube y cuánto es sobreajuste con solo ~100 consultas? usa el bootstrap).
2. **Plegado de tildes** (`tokenize_simple(..., accents='fold')`): ¿cambia el nDCG en ES? ¿Cuándo daña (p. ej. «sí»/«si», «papá»/«papa»)?
3. **Vectoriza** `scores` con `scipy.sparse` (matriz término-documento) y compara velocidad con el índice invertido en `FAST_DEV_RUN=False`.
4. **Golden set honesto.** Construye 20 consultas que *no* copien el vocabulario del documento (parafraseadas a mano) y mide cuánto cae BM25 frente a las de «ítem conocido».
5. **IDF Lucene vs Okapi**: mide la diferencia de nDCG@10 y de rankings (tau de Kendall) entre ambas variantes.

## Reflexión
* El baseline de código es el más débil. Con lo que sabes del pipeline texto → bytes → tokens → embeddings, ¿qué **dos** módulos del curso atacan ese problema y por qué?
* ¿Qué decisión de **tokenización léxica** (acentos, mayúsculas, división de identificadores) tomarías en Políglota en producción y cómo la validarías sin sobreajustar a tu golden set?
* Si mañana cambias la normalización Unicode (módulo 01) de tus documentos, ¿qué tienes que volver a calcular? ¿Y si cambias el tokenizer del embedder (módulo 02)?

---
# ⛔ SPOILER — intenta resolverlo primero
Debajo está la solución de referencia completa y ejecutable. Si «Ejecutar todo» llega hasta aquí, estas celdas **sobrescriben** tus funciones con las de referencia (es útil para comprobar la rúbrica con resultados correctos).

In [ ]:
# ── SOLUCIÓN · TODO 1 ──
def estadisticas_corpus(docs):
    df = pd.DataFrame(docs)
    df["chars"] = df.text.str.len()
    df["bytes"] = df.text.map(lambda t: len(t.encode("utf-8")))
    df["palabras"] = df.text.str.split().map(len)
    g = df.groupby("lang").agg(n_docs=("id", "count"), chars_medio=("chars", "mean"),
                               bytes_medio=("bytes", "mean"), palabras_medio=("palabras", "mean"))
    g["bytes_por_char"] = g.bytes_medio / g.chars_medio
    return g.round(3)

In [ ]:
# ── SOLUCIÓN · TODO 2 ──
def dcg_at_k(ranked_ids, rels, k=10):
    ranked = list(dict.fromkeys(ranked_ids))[:k]                      # dedup conservando orden
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)

def ndcg_at_k(ranked_ids, rels, k=10):
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg

def recall_at_k(ranked_ids, rels, k=10):
    relevantes = {d for d, r in rels.items() if r > 0}
    if not relevantes:
        return 0.0
    return len(relevantes & set(list(dict.fromkeys(ranked_ids))[:k])) / len(relevantes)

def mrr_at_k(ranked_ids, rels, k=10):
    for i, d in enumerate(list(dict.fromkeys(ranked_ids))[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0

In [ ]:
# ── SOLUCIÓN · TODO 3 ──
class BM25:
    def __init__(self, k1=1.2, b=0.75, idf="lucene", epsilon=0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens, doc_ids):
        self.doc_ids, self.N = list(doc_ids), len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean())
        self.postings = defaultdict(list)                                  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        n = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - c + 0.5) / (c + 0.5)) for t, c in n.items()}
        else:
            self.idf = {t: math.log(self.N - c + 0.5) - math.log(c + 0.5) for t, c in n.items()}
            piso = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else piso) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens):
        s = np.zeros(self.N)
        for term in q_tokens:
            for i, tf in self.postings.get(term, ()):
                norma = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norma)
        return s

In [ ]:
# ── SOLUCIÓN · TODO 4 ──
def split_ident(text):
    """'getHTTPResponse_code' -> ['get', 'http', 'response', 'code'] (+ el identificador entero si tenía '_')."""
    enteros = [w for w in tokenize_simple(text) if "_" in w]
    t = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", text)
    t = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1 \2", t).replace("_", " ")
    return tokenize_simple(t) + enteros

In [ ]:
# ── SOLUCIÓN · TODO 5, 6, 7, 8 ──
def pipeline(tok_doc):
    bm = BM25().fit([tok_doc(d) for d in docs], [d["id"] for d in docs])
    return bm, {q["qid"]: buscar(bm, tokenize_simple(q["text"]), 10) for q in queries}

bm_base, run_base = pipeline(lambda d: tokenize_simple(d["text"]))
bm_split, run_split = pipeline(lambda d: split_ident(d["text"]) if d["lang"] == "code" else tokenize_simple(d["text"]))
RESULTADOS = {"ndcg_all": evaluar_grupo(run_split), "ndcg_es": evaluar_grupo(run_split, "es"), "ndcg_en": evaluar_grupo(run_split, "en"),
              "ndcg_code": evaluar_grupo(run_split, "code"), "ndcg_code_base": evaluar_grupo(run_base, "code")}

def bootstrap_ci(valores, n_boot=2000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    medias = rng.choice(valores, size=(n_boot, len(valores)), replace=True).mean(axis=1)
    return float(valores.mean()), tuple(np.quantile(medias, [alpha / 2, 1 - alpha / 2]))

v = np.array([ndcg_at_k(run_split[q["qid"]], qrels[q["qid"]]) for q in queries])
m, (lo, hi) = bootstrap_ci(v); RESULTADOS["ci95"] = [float(lo), float(hi)]

def peores_consultas(bm, tok_q=tokenize_simple, lang="code", n=5):
    out = []
    for q in (q for q in queries if q["lang"] == lang):
        rank = buscar(bm, tok_q(q["text"]), len(docs))
        rel = next(iter(qrels[q["qid"]]))
        out.append((q["qid"], q["text"], rank.index(rel) + 1))
    return sorted(out, key=lambda t: -t[2])[:n]

def guardar_baseline(path="poliglota_v0.json"):
    d = {"source": source, "n_docs": len(docs), "n_queries": len(queries), "k1": bm_split.k1, "b": bm_split.b,
         "tokenizacion": "tokenize_simple (+ split_ident en código)", **{k: RESULTADOS[k] for k in RESULTADOS}}
    pathlib.Path(path).write_text(json.dumps(d, indent=2, ensure_ascii=False))
    return d

print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in RESULTADOS.items()})
print(f"nDCG@10 global = {m:.3f}  IC95% [{lo:.3f}, {hi:.3f}]")
for qid, texto, pos in peores_consultas(bm_split):
    print(f"  pos={pos:>4} | {texto[:90]!r}")
chequea("R1 estadísticas", test_stats); chequea("R2 métricas", test_metricas); chequea("R3a BM25 mini", test_bm25)
chequea("R3 BM25 == rank_bm25", test_vs_rank_bm25); chequea("R6a split_ident", test_split); chequea("R7a bootstrap", test_ci)
chequea("R7b poliglota_v0.json", test_json)
rubrica()

> **📌 Valores de referencia de la solución.** Con corpus **sintético**, `FAST_DEV_RUN=True`, semilla 42 (oct-2026): nDCG@10 global 0.878, ES 0.982, EN 0.932, código 0.668 (0.606 con `\w+`). Con datos **reales** tus números serán distintos (no pudimos medirlos en el entorno de autoría); la celda anterior imprime los de tu ejecución. Guarda `poliglota_v0.json`: el módulo 15 lo usará como punto de partida.